# SNN Single Run — Colab Runner (`learning/main.py`)

Runs `learning/main.py` directly: **one** framework on **one** dataset, the exact same pipeline that runs locally (encoder + adaptive cache -> training -> testing -> adversarial-robustness eval), producing `outputs/data/training_results.csv`, `outputs/data/test.csv`, and `outputs/data/adversarial_robustness.csv`.

This is different from the other notebooks in this folder (`run_on_colab*.ipynb`), which loop `docs/results/run_benchmark.py` across all 4 frameworks. This one is for a single real run — including a fast sanity-check mode (`QUICK_TEST`, see step 3) to confirm the pipeline runs end-to-end before committing to a full multi-epoch run.

No Google Drive mounting — everything stays on this Colab session's local disk (`/content/...`). Download `outputs/` (last cell) before the session ends or the results are gone.

**Runtime -> Change runtime type -> GPU**, before running anything below.

## 1. Get the codebase

Clones the branch this work is actually on. If your remote/branch differs, edit the URL/branch below before running.

In [ ]:
REPO_URL = "https://github.com/Zuzu3290/SNNs-auf-GPUs.git"
BRANCH = "46-cache_engine"  # main is behind and doesn't have this work yet

!git clone --branch {BRANCH} {REPO_URL} /content/SNNs-auf-GPUs
%cd /content/SNNs-auf-GPUs

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print("torch:", torch.__version__, " CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — check Runtime > Change runtime type > GPU")

## 3. Configure the run

`main.py` has no CLI flags — it reads `configuration/SNN_module.yaml`. This cell edits that file in place before the run.

- `FRAMEWORK`: `norse` | `torch` | `sj` | `sinabs`
- `DATASET`: `N-MNIST` | `N-Caltech101` | `DVS128 Gesture` — **classification only**. `DAVIS Camera Pose` and `DSEC` are regression datasets; `main.py` deliberately rejects them (classification-only pipeline).
- `QUICK_TEST = True`: 1 epoch, 5 iterations — just enough to confirm the encoder, cache, model, trainer, tester, and adversarial evaluator all run end-to-end without error. Takes a few minutes.
- `QUICK_TEST = False`: uses the repo's committed defaults (5 epochs x 400 iterations/epoch) for a real result.

In [ ]:
import yaml

FRAMEWORK = "norse"
DATASET = "N-MNIST"
QUICK_TEST = True

cfg_path = "configuration/SNN_module.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg["training"]["framework"] = FRAMEWORK
cfg["dataset"]["dataset_name"] = DATASET
if QUICK_TEST:
    cfg["training"]["epochs"] = 1
    cfg["training"]["iterations_per_epoch"] = 5

with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"framework={FRAMEWORK}  dataset={DATASET}  "
      f"epochs={cfg['training']['epochs']}  iterations_per_epoch={cfg['training']['iterations_per_epoch']}")

## 4. Run main.py

In [ ]:
!python learning/main.py

## 5. View results

In [ ]:
import pandas as pd
from pathlib import Path

for name in ["training_results.csv", "test.csv", "adversarial_robustness.csv"]:
    path = Path("outputs/data") / name
    if path.exists():
        print(f"
=== {path} ===")
        display(pd.read_csv(path))
    else:
        print(f"
=== {path} — not found (run may not have reached this stage) ===")

## 6. Download results before the session ends

No Drive mounting — zips `outputs/` and `checkpoints/` and triggers a browser download instead.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/snn_main_run_results", "zip", ".", "outputs")
files.download("/content/snn_main_run_results.zip")